# Lezione 10 — Monitoraggio

**Corso:** MLOps e Monitoraggio con Python · **Blocco B**
**Tempo stimato:** 100 minuti · **Capitolo del manuale:** `manual/10_monitorizacion_IT.md`

## Cosa vedrai in questa lezione

- La differenza tra **metriche di sistema** (latenza, tasso di errore, uso di CPU/memoria) e **metriche di modello** (distribuzione delle previsioni, un proxy di qualità quando non c'è un'etichetta immediata).
- Come strumentare l'API di churn della lezione 7 con **logging strutturato** reale (il modulo `logging` di Python, con un formatter JSON), invece di `print`.
- Come simulare, con un seed fisso, un giorno completo di traffico con **2 incidenti sintetici** inseriti apposta (un picco di latenza, un'ondata di errori).
- Come costruire un **pannello** che riassuma queste metriche per finestra oraria, con una tabella di `pandas` (la fonte verificabile di ogni cifra) e un grafico di `matplotlib` salvato su disco.
- Come definire **allarmi** basati su soglie che rilevino i 2 incidenti reali senza falsi positivi, e il compromesso concreto tra allarmare troppo e troppo poco.

Tutto avviene dentro una cartella di lavoro effimera (`proyecto_churn/`), creata all'inizio ed **eliminata alla fine** di questo stesso notebook. Nessun "tempo di risposta" di questo notebook viene misurato con l'orologio reale: tutti i tempi sono valori sintetici, generati con un seed fisso, perché la lezione sia identica ad ogni esecuzione.

## Prerequisiti

- Dalla lezione 7: l'API FastAPI (`Cliente`, `/predecir`, `TestClient`), che questo notebook ricostruisce e amplia con il logging.
- Dalle lezioni 4 e 5: la configurazione vincitrice del modello (regressione logistica, `class_weight="balanced"`, `C=1,0`).
- Python di base: funzioni, dizionari, il modulo `logging` della libreria standard (questa lezione lo introduce da zero).

> **Come usare questo notebook:** esegui le celle in ordine, dall'alto in basso, una sola volta. Richiede meno di un minuto in totale.

---
## Installazione

Installa solo ciò che manca (FastAPI, httpx, matplotlib).

In [ ]:
import warnings

# Silenziamo un avviso di compatibilità di starlette/httpx (fisso, non aggiunge nulla qui) perché
# l'output di questo notebook non dipenda da dettagli interni di versioni delle librerie
warnings.filterwarnings("ignore", category=DeprecationWarning)

import subprocess
import sys


def asegurar(paquete, modulo):
    """Installa il pacchetto solo se manca (idempotente e silenzioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

asegurar("numpy", "numpy")
asegurar("pandas", "pandas")
asegurar("scikit-learn", "sklearn")
asegurar("matplotlib", "matplotlib")
asegurar("joblib", "joblib")
asegurar("fastapi", "fastapi")
asegurar("httpx", "httpx")
print("Librerie pronte.")

---
## Cella 0 — Setup

In [ ]:
import os
import shutil
import json
import logging
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score

# Silenziamo un avviso di compatibilità di starlette/httpx (fisso, non aggiunge nulla qui) perché
# l'output di questo notebook non dipenda da dettagli interni di versioni delle librerie
warnings.filterwarnings("ignore", category=DeprecationWarning)

def cargar_datos(nombre):
    """Cerca un file di dati in datos/, ../datos/ e /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/18-mlops-monitoraggio/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "Non trovo " + nombre + ". Carica la cartella datos/ del corso (in Colab: pannello dei file) o lasciala accanto al notebook.")

SEMILLA = 42
COLUMNAS_CATEGORICAS = ["plan", "contrato", "metodo_pago"]
COLUMNAS_NUMERICAS = ["antiguedad_meses", "uso_mensual_horas", "tickets_soporte", "factura_mensual"]
COLUMNAS_ENTRADA = COLUMNAS_CATEGORICAS + COLUMNAS_NUMERICAS

print("Librerie e funzioni ausiliarie pronte.")

---
## Cartella di lavoro effimera (si pulisce all'inizio e alla fine)

In [ ]:
CARPETA = "proyecto_churn"

# Puliamo e ricreiamo la cartella perché il notebook sia ripetibile da zero (anche all'inizio, nel caso sia rimasto qualcosa da un'esecuzione precedente)
if os.path.exists(CARPETA):
    shutil.rmtree(CARPETA)
os.makedirs(os.path.join(CARPETA, "logs"))
print("Cartella di lavoro creata:", os.path.abspath(CARPETA))

---
## 1. Riaddestrare il modello e ricostruire l'API della lezione 7

Questa lezione non dipende da nessun file salvato in una lezione precedente: riaddestra la stessa Pipeline delle lezioni 4, 5 e 7, e ricostruisce un unico endpoint (`/predecir`) dell'API FastAPI della lezione 7 -- sufficiente per generare traffico da monitorare.

In [ ]:
RUTA_CSV = cargar_datos("churn.csv")
df = pd.read_csv(RUTA_CSV)

X = df[COLUMNAS_ENTRADA]
y = df["abandono"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEMILLA, stratify=y
)

# Stessa configurazione vincitrice delle lezioni 4, 5 e 7: regressione logistica, class_weight="balanced", C=1.0
preprocesador = ColumnTransformer([
    ("categoricas", OneHotEncoder(handle_unknown="ignore"), COLUMNAS_CATEGORICAS),
    ("numericas", StandardScaler(), COLUMNAS_NUMERICAS),
])
clasificador = LogisticRegression(class_weight="balanced", random_state=SEMILLA, max_iter=1000, C=1.0)
pipeline = Pipeline([("preprocesado", preprocesador), ("clasificador", clasificador)])
pipeline.fit(X_train, y_train)

f1 = round(f1_score(y_test, pipeline.predict(X_test)), 4)
auc = round(roc_auc_score(y_test, pipeline.predict_proba(X_test)[:, 1]), 4)
print("F1 (set di test):", f1)
print("AUC (set di test):", auc)
assert f1 > 0.25 and auc > 0.6, "il ri-addestramento non ha riprodotto il modello atteso"

RUTA_MODELO = os.path.join(CARPETA, "modelo.pkl")
joblib.dump(pipeline, RUTA_MODELO)
MODELO = joblib.load(RUTA_MODELO)
assert np.array_equal(MODELO.predict(X_test), pipeline.predict(X_test))
print("Modello riaddestrato (come nella lezione 7) e salvato come artefatto in", RUTA_MODELO)

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal
from fastapi import FastAPI
from fastapi.testclient import TestClient


class Cliente(BaseModel):
    """Lo stesso modello Pydantic della lezione 7: le colonne di input di churn.csv."""
    id_cliente: str
    antiguedad_meses: int = Field(ge=0)
    plan: Literal["basico", "estandar", "premium"]
    contrato: Literal["mensual", "anual"]
    uso_mensual_horas: float = Field(ge=0)
    tickets_soporte: int = Field(ge=0)
    metodo_pago: Literal["domiciliacion", "tarjeta", "transferencia"]
    factura_mensual: float = Field(ge=0)


def cliente_a_fila(cliente):
    datos = cliente.model_dump()
    return pd.DataFrame([datos])[COLUMNAS_ENTRADA]


app = FastAPI(title="API di churn (lezione 7, ampliata con monitoraggio in questa lezione)")


@app.post("/predecir")
def predecir(cliente: Cliente):
    fila = cliente_a_fila(cliente)
    probabilidad = float(MODELO.predict_proba(fila)[0, 1])
    return {
        "id_cliente": cliente.id_cliente,
        "probabilidad_abandono": round(probabilidad, 4),
        "abandono_previsto": int(probabilidad >= 0.5),
    }


cliente_api = TestClient(app)
print("API della lezione 7 ricostruita (un solo endpoint, /predecir, è sufficiente per questa lezione). Pronta per essere strumentata con il logging.")

---
## 2. Logging strutturato: un formatter JSON reale

Invece di `print`, definiamo un `logging.Formatter` che convertisce ogni evento in una riga JSON con 7 campi sempre presenti (`timestamp`, `nivel`, `endpoint`, `id_cliente`, `status_code`, `latencia_ms`, `probabilidad_abandono`). Il campo `timestamp` è un valore che passiamo NOI esplicitamente (con `extra=...`), non l'ora reale che `logging` catturerebbe di default -- così l'intero log è riproducibile.

In [ ]:
class FormateadorJSON(logging.Formatter):
    """Formatta ogni LogRecord come una riga JSON con campi coerenti, invece di un messaggio di testo libero.

    Il campo "timestamp" proviene da un valore SINTETICO che passiamo noi con extra=..., non dall'orologio reale del sistema (record.created): così l'intero log è riproducibile con un seed fisso, cosa che non potremmo garantire con l'ora reale di esecuzione.
    """

    def format(self, record):
        registro = {
            "timestamp": getattr(record, "timestamp", None),
            "nivel": record.levelname,
            "endpoint": getattr(record, "endpoint", None),
            "id_cliente": getattr(record, "id_cliente", None),
            "status_code": getattr(record, "status_code", None),
            "latencia_ms": getattr(record, "latencia_ms", None),
            "probabilidad_abandono": getattr(record, "probabilidad_abandono", None),
            "mensaje": record.getMessage(),
        }
        return json.dumps(registro, ensure_ascii=False)


RUTA_LOG = os.path.join(CARPETA, "logs", "peticiones.log")

logger_api = logging.getLogger("api_churn")
logger_api.setLevel(logging.INFO)
logger_api.handlers.clear()  # evita di duplicare gli handler se questa cella viene eseguita più di una volta
manejador = logging.FileHandler(RUTA_LOG, mode="w", encoding="utf-8")
manejador.setFormatter(FormateadorJSON())
logger_api.addHandler(manejador)
logger_api.propagate = False


def registrar_peticion(timestamp, endpoint, id_cliente, status_code, latencia_ms, probabilidad_abandono=None):
    """Registra una richiesta con logging STRUTTURATO reale (non print): ogni chiamata aggiunge una riga JSON al file di log, con gli stessi campi sempre nello stesso posto."""
    logger_api.info(
        "peticion",
        extra={
            "timestamp": timestamp, "endpoint": endpoint, "id_cliente": id_cliente,
            "status_code": status_code, "latencia_ms": round(latencia_ms, 3),
            "probabilidad_abandono": None if probabilidad_abandono is None else round(probabilidad_abandono, 4),
        },
    )


print("Logger strutturato pronto. Ogni riga scritta in", RUTA_LOG, "sarà JSON valido, con gli stessi 7 campi sempre.")

---
## 3. Una demo con l'API reale

Prima di simulare un giorno intero, tre richieste reali tramite `TestClient`, registrate con `registrar_peticion`. Le "latenze" di questa demo sono valori fissi di esempio (non misurati con l'orologio): questa lezione distingue esplicitamente "verificare che l'API risponda" (qui) da "misurare quanto impiega" (lezione 7, con l'eccezione dichiarata di quella lezione; in questa lezione 10 non misuriamo mai il tempo reale).

In [ ]:
# Tre richieste di esempio, con latenze SINTETICHE fisse (non misurate con time.perf_counter): in questa lezione
# NESSUN "tempo di risposta" viene misurato con l'orologio reale -- tutti, inclusa questa demo, sono valori di esempio predefiniti, perché il notebook sia identico ad ogni esecuzione
cliente_demo = {
    "id_cliente": "C_DEMO_0001", "antiguedad_meses": 12, "plan": "estandar", "contrato": "mensual",
    "uso_mensual_horas": 15.0, "tickets_soporte": 1, "metodo_pago": "tarjeta", "factura_mensual": 22.5,
}

respuesta_ok = cliente_api.post("/predecir", json=cliente_demo)
registrar_peticion("2026-01-12T09:00:00", "/predecir", cliente_demo["id_cliente"], respuesta_ok.status_code, 12.3,
                    respuesta_ok.json()["probabilidad_abandono"])

cliente_invalido = dict(cliente_demo, plan="diamante")
respuesta_422 = cliente_api.post("/predecir", json=cliente_invalido)
registrar_peticion("2026-01-12T09:00:01", "/predecir", cliente_invalido["id_cliente"], respuesta_422.status_code, 3.1)

respuesta_lenta = cliente_api.post("/predecir", json=cliente_demo)
registrar_peticion("2026-01-12T09:00:02", "/predecir", cliente_demo["id_cliente"], respuesta_lenta.status_code, 181.4,
                    respuesta_lenta.json()["probabilidad_abandono"])

with open(RUTA_LOG, encoding="utf-8") as f:
    lineas_demo = f.readlines()
print("Righe scritte finora:", len(lineas_demo))
for linea in lineas_demo:
    print(linea.strip())
    json.loads(linea)  # conferma che ogni riga è JSON valido

---
## 4. Simulare un giorno di traffico con 2 incidenti sintetici

`generar_registro_dia` percorre 24 finestre orarie e, per ciascuna, un numero variabile di richieste (tra 30 e 70), tutto con lo stesso `SEMILLA`. Nell'ora 14 la latenza si alza (media ≈160 ms invece di ≈12 ms); nell'ora 20 il tasso di errore si alza (≈32 % invece di ≈1,2 %). Ogni richiesta viene registrata con `registrar_peticion`: il log risultante è la fonte di dati di tutto il resto della lezione.

> **Attenzione:** nessuna latenza di questa sezione è misurata con `time.perf_counter()` né con nessun orologio reale. Sono valori generati con `numpy.random.default_rng(SEMILLA)`, esattamente riproducibili.

In [ ]:
# Finestre orarie con incidente sintetico inserito apposta -- NON si scelgono a caso ad ogni esecuzione,
# sono costanti fisse dell'esercizio, esattamente come richiede l'enunciato della lezione 10
HORA_INCIDENTE_LATENCIA = 14   # picco di latenza tra le 14:00 e le 15:00
HORA_INCIDENTE_ERRORES = 20    # ondata di errori tra le 20:00 e le 21:00

LATENCIA_BASE_MEDIA_MS, LATENCIA_BASE_DESV_MS = 12.0, 2.5
LATENCIA_INCIDENTE_MEDIA_MS, LATENCIA_INCIDENTE_DESV_MS = 160.0, 30.0
TASA_ERROR_BASE = 0.012
TASA_ERROR_INCIDENTE = 0.32
FECHA_BASE = "2026-01-12"

rng = np.random.default_rng(SEMILLA)


def generar_registro_dia():
    """Genera, con il seed fisso SEMILLA, un giorno completo (24 finestre orarie) di richieste simulate a /predecir, con 2 incidenti sintetici inseriti apposta. Ogni richiesta viene registrata con logging strutturato reale (registrar_peticion), non con print."""
    total = 0
    for hora in range(24):
        n_peticiones = int(rng.integers(30, 71))
        for i in range(n_peticiones):
            idx = rng.integers(0, len(df))
            fila = df.iloc[[idx]][COLUMNAS_ENTRADA]
            probabilidad = float(MODELO.predict_proba(fila)[0, 1])

            if hora == HORA_INCIDENTE_LATENCIA:
                latencia = float(rng.normal(LATENCIA_INCIDENTE_MEDIA_MS, LATENCIA_INCIDENTE_DESV_MS))
            else:
                latencia = float(rng.normal(LATENCIA_BASE_MEDIA_MS, LATENCIA_BASE_DESV_MS))
            latencia = max(latencia, 0.1)  # una latenza non può essere negativa

            p_error = TASA_ERROR_INCIDENTE if hora == HORA_INCIDENTE_ERRORES else TASA_ERROR_BASE
            es_error = rng.random() < p_error
            status_code = 500 if es_error else 200

            minuto = int(i * 60 / n_peticiones)
            timestamp = f"{FECHA_BASE}T{hora:02d}:{minuto:02d}:00"
            id_cliente = f"C_SIM_{hora:02d}_{i:04d}"
            registrar_peticion(timestamp, "/predecir", id_cliente, status_code, latencia, probabilidad)
            total += 1
    return total


N_REGISTRADOS = generar_registro_dia()
print("Richieste simulate e registrate nel log:", N_REGISTRADOS)
assert N_REGISTRADOS > 3 * 24, "ci aspettavamo diverse decine di richieste per ora"

---
## 5. Leggere il log strutturato (come farebbe un sistema reale di analisi)

In [ ]:
registros = []
with open(RUTA_LOG, encoding="utf-8") as f:
    for linea in f:
        registros.append(json.loads(linea))

df_log = pd.DataFrame(registros)
df_log["hora"] = df_log["timestamp"].str.slice(11, 13).astype(int)
print("Righe totali nel log (incluse le 3 richieste di demo):", len(df_log))
print(df_log.head(3).to_string(index=False))

---
## 6. Il pannello: una tabella aggregata per finestra oraria

`panel` è una tabella `pandas` con una riga per ora: numero di richieste, tasso di errore, latenza media, latenza p95 e la probabilità media prevista (una **metrica di modello**: un proxy di qualità, utile quando -- come in produzione -- non abbiamo l'etichetta reale `abandono` di un cliente al momento della previsione).

In [ ]:
# Pannello: una riga per finestra oraria, con le metriche di SISTEMA (richieste, tasso di errore, latenza)
# e una metrica di MODELLO (probabilità media prevista): un proxy di qualità che non ha bisogno di conoscere l'etichetta reale
df_peticiones_simuladas = df_log[df_log["id_cliente"].str.startswith("C_SIM_")].copy()

panel = (
    df_peticiones_simuladas.groupby("hora")
    .agg(
        n_peticiones=("status_code", "size"),
        tasa_error=("status_code", lambda s: (s >= 500).mean()),
        latencia_media_ms=("latencia_ms", "mean"),
        latencia_p95_ms=("latencia_ms", lambda s: s.quantile(0.95)),
        probabilidad_media=("probabilidad_abandono", "mean"),
    )
    .reset_index()
)
pd.set_option("display.width", 140)
print(panel.round(4).to_string(index=False))
assert len(panel) == 24, "il pannello deve avere esattamente una riga per ciascuna delle 24 ore"

**Cosa osservi?** Le ore 14 e 20 spiccano chiaramente nella tabella: l'ora 14 ha una `latencia_media_ms` quasi 13 volte maggiore delle ore normali, e l'ora 20 ha una `tasa_error` circa 27 volte maggiore. La `probabilidad_media` (metrica di modello), invece, resta in un intervallo simile nelle 24 ore -- in questo log non simuliamo nessun cambiamento nella distribuzione delle previsioni, solo incidenti di sistema. È esattamente ciò che la lezione 11 (*drift*) indagherà: cosa fare quando quella distribuzione CAMBIA davvero.

---
## 7. Lo stesso pannello, come grafico (salvato, non mostrato)

In [ ]:
# Salviamo il grafico su disco ma NON lo mostriamo (plt.show()): il testo della tabella sopra
# è la fonte verificabile di ogni numero; il grafico è solo un supporto visivo complementare
figura, (ax_latencia, ax_error) = plt.subplots(2, 1, figsize=(9, 6), sharex=True)

ax_latencia.plot(panel["hora"], panel["latencia_media_ms"], marker="o", color="tab:blue")
ax_latencia.set_ylabel("Latenza media (ms)")
ax_latencia.set_title("Metriche di sistema per finestra oraria (giorno simulato)")
ax_latencia.grid(alpha=0.3)

ax_error.plot(panel["hora"], panel["tasa_error"] * 100, marker="o", color="tab:red")
ax_error.set_ylabel("Tasso di errore (%)")
ax_error.set_xlabel("Ora del giorno")
ax_error.grid(alpha=0.3)

RUTA_GRAFICO = os.path.join(CARPETA, "panel_monitorizacion.png")
figura.tight_layout()
figura.savefig(RUTA_GRAFICO, dpi=100)
plt.close(figura)
print("Grafico salvato in:", RUTA_GRAFICO)
assert os.path.exists(RUTA_GRAFICO)

---
## 8. Allarmi basati su soglie

`detectar_incidentes` segnala una finestra oraria se la sua latenza media o il suo tasso di errore superano una soglia. Con le soglie di produzione scelte (50 ms, 10 %), rileva ESATTAMENTE le ore 14 e 20 -- non un'ora normale in più, non una in meno.

In [ ]:
def detectar_incidentes(panel_agregado, umbral_latencia_ms, umbral_tasa_error):
    """Restituisce la lista ORDINATA delle ore la cui latenza media o il cui tasso di errore superano la soglia data. È la definizione completa di un allarme basato su soglie."""
    supera = (
        (panel_agregado["latencia_media_ms"] > umbral_latencia_ms)
        | (panel_agregado["tasa_error"] > umbral_tasa_error)
    )
    return sorted(panel_agregado.loc[supera, "hora"].tolist())


# Soglie di produzione: si scelgono guardando il pannello -- molto sopra il comportamento normale
# (latenza media normale ≈12 ms, tasso di errore normale ≈0,9 %) e molto sotto i due incidenti reali
# (≈154 ms di latenza, ≈24 % di errori): un margine ampio da entrambi i lati
UMBRAL_LATENCIA_MS = 50.0
UMBRAL_TASA_ERROR = 0.10

incidentes_reales = {HORA_INCIDENTE_LATENCIA, HORA_INCIDENTE_ERRORES}
incidentes_detectados = detectar_incidentes(panel, UMBRAL_LATENCIA_MS, UMBRAL_TASA_ERROR)
print("Ore segnalate dall'allarme con le soglie di produzione:", incidentes_detectados)

# Verifica concreta: rileva ESATTAMENTE i 2 incidenti inseriti, senza falsi positivi nelle 22 ore normali
assert set(incidentes_detectados) == incidentes_reales, "l'allarme doveva segnalare esattamente le ore 14 e 20"
assert len(incidentes_detectados) == len(incidentes_reales), "non doveva esserci nessun falso positivo con questa soglia"
print("I 2 incidenti sintetici vengono rilevati correttamente e non c'è nessun falso positivo tra le 22 ore normali.")

---
## 9. Il compromesso: allarmare troppo contro allarmare troppo poco

Su questo stesso log, verifichiamo i due estremi del compromesso con numeri concreti: una soglia troppo aggressiva genera falsi positivi (fatica da allarme); una soglia troppo alta perde gli incidenti reali.

In [ ]:
# Soglia troppo AGGRESSIVA (troppo bassa): "allarmare troppo" -- fatica da allarme
UMBRAL_LATENCIA_AGRESIVO = 13.0
UMBRAL_TASA_ERROR_AGRESIVO = 0.02
incidentes_agresivo = detectar_incidentes(panel, UMBRAL_LATENCIA_AGRESIVO, UMBRAL_TASA_ERROR_AGRESIVO)
falsos_positivos_agresivo = sorted(set(incidentes_agresivo) - incidentes_reales)
print("Con una soglia aggressiva (latenza > 13 ms o errore > 2 %), ore segnalate:", incidentes_agresivo)
print("Di queste, falsi positivi (ore normali segnalate per errore):", len(falsos_positivos_agresivo), falsos_positivos_agresivo)
assert len(falsos_positivos_agresivo) > 0, "la soglia aggressiva doveva produrre almeno un falso positivo, per illustrare la fatica da allarme"

# Soglia troppo ALTA: "allarmare troppo poco" -- gli incidenti reali passano inosservati
UMBRAL_LATENCIA_ALTO = 500.0
UMBRAL_TASA_ERROR_ALTO = 0.60
incidentes_umbral_alto = detectar_incidentes(panel, UMBRAL_LATENCIA_ALTO, UMBRAL_TASA_ERROR_ALTO)
incidentes_perdidos = sorted(incidentes_reales - set(incidentes_umbral_alto))
print("Con una soglia troppo alta (latenza > 500 ms o errore > 60 %), ore segnalate:", incidentes_umbral_alto)
print("Dei 2 incidenti reali, quanti si perdono completamente:", len(incidentes_perdidos), incidentes_perdidos)
assert len(incidentes_perdidos) == 2, "la soglia troppo alta doveva perdere i 2 incidenti reali, per illustrare l'altro estremo del compromesso"

print()
print("Riepilogo del compromesso, con le cifre di questo stesso log:")
print("-- soglia aggressiva: 0 incidenti persi, ma", len(falsos_positivos_agresivo), "falsi positivi su 22 ore normali")
print("-- soglia di produzione (scelta in questa lezione): 0 falsi positivi e 0 incidenti persi")
print("-- soglia troppo alta: 0 falsi positivi, ma", len(incidentes_perdidos), "incidenti reali persi su 2")

**Cosa osservi?** Con le cifre ricalcolate in questo stesso notebook: la soglia aggressiva (13 ms / 2 %) segnala diverse ore normali come incidenti -- puro rumore statistico della variazione ora per ora, non un problema reale -- mentre la soglia di produzione (50 ms / 10 %) non commette questo errore perché lascia un margine ampio sopra la variazione normale. La soglia troppo alta (500 ms / 60 %), all'altro estremo, non segnala NULLA: i 2 incidenti reali di questo log passerebbero inosservati a chiunque guardasse solo quel pannello di allarmi. Nessuna soglia è "quella corretta" in assoluto: dipende da quanto rumore di fondo ha il tuo sistema e da quanto costa (in tempo di qualcuno) revisionare un falso allarme rispetto a quanto costa non vedere un incidente reale.

---
## 10. Riepilogo e chiusura

- Abbiamo distinto le metriche di **sistema** (latenza, tasso di errore) dalle metriche di **modello** (distribuzione delle previsioni, un proxy senza etichetta immediata).
- Abbiamo strumentato l'API della lezione 7 con **logging strutturato** reale (`logging` + un formatter JSON), non `print`.
- Abbiamo simulato, con seed fisso, un giorno di traffico con 2 incidenti sintetici, e costruito un pannello (tabella + grafico salvato) che li rende visibili.
- Abbiamo definito allarmi per soglia che rilevano i 2 incidenti reali senza falsi positivi, e misurato con numeri concreti il compromesso tra soglie troppo aggressive e troppo alte.

*Continua con la lezione 11: Drift e riaddestramento.*

---
## Pulizia finale

In [ ]:
shutil.rmtree(CARPETA)
print("Cartella di lavoro eliminata:", CARPETA)
assert not os.path.exists(CARPETA)